# servo_bus: Feetech STS3215 packets and the bus driver

Library notebook for `gambit_arm`. The SO-ARM101 uses six Feetech STS3215 serial bus servos
on one half-duplex TTL line, driven through the Waveshare Bus Servo Adapter (USB serial).
This notebook builds and parses SCS/STS protocol packets (pure bytes, fully tested here) and
sketches a `ServoBus` class on top of pyserial (I/O left as TODO, needs the hardware).

The same protocol is implemented in C++ (`gambit_cpp/libs/servo_bus/sts_protocol.cpp`) and
C (`firmware/stm32/Core/Src/servo_bus.c`); all three must produce identical bytes. The
self-check vectors below are also in `gambit_cpp/tests/test_sts_protocol.cpp`.

Milestone: *Hardware and execution* (servo IDs assigned, calibrated, teleoperating).

**TODO**
- [ ] packet encode (PING / READ / WRITE / SYNC_WRITE) and checksum
- [ ] status packet decode with checksum check
- [ ] ticks <-> radians (4096 ticks per revolution)
- [ ] verify every control-table address against the STS3215 datasheet (marked VERIFY)
- [ ] `ServoBus` I/O: open port, transact with timeout, handle half-duplex echo
- [ ] per-joint calibration (offset and direction) from run_servo_setup.ipynb
- [ ] SYNC_READ of present positions (one round trip for all six servos)
- [ ] measure the round-trip time at 1 Mbaud (needed for the Python vs C++ comparison)
- [ ] write the self-checks at the bottom as asserts

In [ ]:
import math
import time
from dataclasses import dataclass, field

## Packet format

```
instruction:  0xFF 0xFF  ID  LEN  INSTR  P1 .. PN  CHK      LEN = N + 2
status:       0xFF 0xFF  ID  LEN  ERROR  P1 .. PN  CHK
CHK = ~(ID + LEN + INSTR/ERROR + P1 + ... + PN) & 0xFF
```

ID 0xFE is broadcast (no reply). STS servos store 16-bit registers **little-endian**
(low byte first); the older SCS series is big-endian, so do not copy SCS examples blindly.

In [ ]:
HEADER = bytes([0xFF, 0xFF])
BROADCAST_ID = 0xFE

class Inst:
    PING = 0x01
    READ = 0x02
    WRITE = 0x03
    REG_WRITE = 0x04
    ACTION = 0x05
    SYNC_READ = 0x82
    SYNC_WRITE = 0x83

# Control table (EEPROM below 0x28, RAM from 0x28). VERIFY against the STS3215 datasheet /
# Feetech memory table before writing anything, especially EEPROM (ID, baud, limits, offset).
class Reg:
    ID = 0x05                   # VERIFY
    BAUD_RATE = 0x06            # VERIFY (0 = 1 Mbaud)
    MIN_ANGLE_LIMIT = 0x09      # 2 bytes, VERIFY
    MAX_ANGLE_LIMIT = 0x0B      # 2 bytes, VERIFY
    POSITION_OFFSET = 0x1F      # 2 bytes, VERIFY
    TORQUE_ENABLE = 0x28        # VERIFY
    ACCELERATION = 0x29         # VERIFY
    GOAL_POSITION = 0x2A        # 2 bytes, VERIFY
    GOAL_TIME = 0x2C            # 2 bytes, VERIFY
    GOAL_SPEED = 0x2E           # 2 bytes, VERIFY
    TORQUE_LIMIT = 0x30         # 2 bytes, 0..1000, VERIFY
    LOCK = 0x37                 # EEPROM write lock, VERIFY
    PRESENT_POSITION = 0x38     # 2 bytes, VERIFY
    PRESENT_SPEED = 0x3A        # 2 bytes, VERIFY
    PRESENT_LOAD = 0x3C         # 2 bytes, VERIFY
    PRESENT_VOLTAGE = 0x3E      # VERIFY
    PRESENT_TEMPERATURE = 0x3F  # VERIFY
    MOVING = 0x42               # VERIFY
    PRESENT_CURRENT = 0x45      # 2 bytes, VERIFY

# Error bits in a status packet (VERIFY).
ERROR_BITS = {0: "voltage", 1: "angle", 2: "overheat", 3: "overele", 5: "overload"}

def checksum(body) -> int:
    """body = ID, LEN, INSTR/ERROR, params (everything after the 0xFF 0xFF header)."""
    # Design: bitwise NOT of the byte sum, truncated to 8 bits (formula in the markdown above).
    # TODO: implement
    raise NotImplementedError

def encode(servo_id: int, instruction: int, params=b"") -> bytes:
    """Build one instruction packet: header, ID, LEN, INSTR, params, checksum."""
    # Design:
    # - LEN counts INSTR + params + CHK, i.e. len(params) + 2.
    # - LEN is one byte, so reject param lists that would overflow it (ValueError).
    # - Checksum covers everything after the 0xFF 0xFF header except itself.
    # TODO: implement
    raise NotImplementedError

def u16_le(v: int) -> bytes:
    """16-bit value -> 2 bytes, low byte first (STS is little-endian)."""
    # Design: mask and shift; 1 line.
    # TODO: implement
    raise NotImplementedError

def from_u16_le(lo: int, hi: int) -> int:
    """Inverse of u16_le."""
    # Design: 1 line.
    # TODO: implement
    raise NotImplementedError

def ping(servo_id: int) -> bytes:
    """PING packet (no params). The reply is an empty status packet."""
    # Design: thin wrapper over encode.
    # TODO: implement
    raise NotImplementedError

def read(servo_id: int, address: int, length: int) -> bytes:
    """READ `length` bytes starting at control-table `address`."""
    # Design: params are [address, length].
    # TODO: implement
    raise NotImplementedError

def write(servo_id: int, address: int, data) -> bytes:
    """WRITE `data` bytes starting at `address`."""
    # Design: params are [address] + data bytes.
    # TODO: implement
    raise NotImplementedError

def sync_write(address: int, data_length: int, entries: dict) -> bytes:
    """entries: {servo_id: data bytes (len == data_length)} -> one broadcast packet."""
    # Design:
    # - Params: [address, data_length], then for each servo: [id] + its data bytes.
    # - Sent to BROADCAST_ID with Inst.SYNC_WRITE: no reply, and all six joints update in the
    #   same bus transaction (why we use it for goal positions: no skew between joints).
    # - Reject entries whose data length differs from data_length (ValueError).
    # TODO: implement
    raise NotImplementedError

@dataclass
class Status:
    id: int
    error: int
    params: bytes

    @property
    def errors(self) -> list:
        """Names of the set error bits (ERROR_BITS), in bit order."""
        # Design: test each bit of self.error against ERROR_BITS.
        # TODO: implement
        raise NotImplementedError

def decode_status(data):
    """Parse the first status packet in `data` (garbage before the header is skipped).
    Returns (Status, bytes_consumed) or (None, 0) if incomplete / bad checksum."""
    # Design:
    # - Find the first 0xFF 0xFF header (bytes.find); anything before it is line noise or echo.
    # - Need at least header + ID + LEN + ERROR + CHK (6 bytes) to read LEN, then the full
    #   packet length follows from LEN; if not all bytes have arrived yet, return (None, 0)
    #   so the caller can read more and retry.
    # - Verify the checksum; a bad one is (None, 0), never a Status with wrong data.
    # - bytes_consumed counts from the start of `data` (including skipped garbage) so the
    #   caller can drop exactly what was used.
    # Decision for you: should a bad checksum skip past that header and keep searching?
    # TODO: implement
    raise NotImplementedError

## Position units

4096 ticks per revolution. We map tick 2048 to 0 rad (servo centered); the per-joint
calibration (offset and sign so that 0 rad matches the URDF zero) is a TODO filled by
run_servo_setup.ipynb and stored in `CALIBRATION` below.

In [ ]:
TICKS_PER_REV = 4096
CENTER_TICK = 2048

def ticks_to_rad(ticks: int) -> float:
    """Raw servo ticks -> radians, CENTER_TICK = 0 rad."""
    # Design: linear map, 2*pi per TICKS_PER_REV, offset by CENTER_TICK.
    # TODO: implement
    raise NotImplementedError

def rad_to_ticks(rad: float) -> int:
    """Radians -> raw ticks, rounded and clamped to 0..TICKS_PER_REV-1."""
    # Design: inverse of ticks_to_rad, round to the nearest tick, then clamp: the servo
    # register cannot represent anything outside 0..4095, so never send it out-of-range values.
    # TODO: implement
    raise NotImplementedError

# LeRobot SO-101 convention: IDs 1..6 in this order (verify after assigning IDs).
JOINT_IDS = {"shoulder_pan": 1, "shoulder_lift": 2, "elbow_flex": 3,
             "wrist_flex": 4, "wrist_roll": 5, "gripper": 6}

@dataclass
class JointCalibration:
    offset_ticks: int = 0      # added to the raw reading so URDF zero reads CENTER_TICK
    sign: int = 1              # +1 or -1 if the servo turns opposite to the URDF axis

# TODO: fill from run_servo_setup.ipynb (save/load as YAML next to board.yaml, e.g.
# gambit_bringup/config/arm_calibration.yaml).
CALIBRATION = {name: JointCalibration() for name in JOINT_IDS}

def joint_rad_from_raw(name: str, raw_ticks: int, cal=CALIBRATION) -> float:
    """Raw reading of joint `name` -> URDF joint angle (rad), using its calibration."""
    # Design: apply the offset in tick space first, convert with ticks_to_rad, then the sign.
    # TODO: implement
    raise NotImplementedError

def raw_from_joint_rad(name: str, rad: float, cal=CALIBRATION) -> int:
    """URDF joint angle (rad) -> raw goal ticks. Exact inverse of joint_rad_from_raw."""
    # Design: undo the steps of joint_rad_from_raw in reverse order (sign, rad_to_ticks, offset).
    # Test: raw -> rad -> raw must return the same tick for every joint.
    # TODO: implement
    raise NotImplementedError

## ServoBus (pyserial)

Thin wrapper: one method per instruction. pyserial is imported lazily so this notebook runs
without it. The adapter is half-duplex: depending on the board you may read back your own
transmitted bytes before the reply ("echo"); `transact` must skip them.

In [ ]:
@dataclass
class ServoBus:
    port: str = "/dev/ttyACM0"
    baud: int = 1_000_000
    timeout_s: float = 0.02
    ser: object = field(default=None, repr=False)

    def open(self):
        """Open the serial port (pyserial, imported here) and return self."""
        # Design: serial.Serial(port, baud, timeout=timeout_s); lazy import so the notebook
        # runs without pyserial. 1 Mbaud is the STS3215 factory default.
        # TODO: implement
        raise NotImplementedError

    def close(self):
        """Close the port if open (safe to call twice)."""
        # TODO: implement
        raise NotImplementedError

    def __enter__(self):
        # Design: context manager = open(); lets callers write `with ServoBus(PORT) as bus:`.
        # TODO: implement
        raise NotImplementedError

    def __exit__(self, *exc):
        # Design: always close, even on exceptions. Consider disabling torque first?
        # TODO: implement
        raise NotImplementedError

    def transact(self, packet: bytes, expect_reply: bool = True):
        """Send one instruction packet and return the decoded Status (or None)."""
        # Design:
        # - Clear stale input (reset_input_buffer), write the packet, flush.
        # - Half-duplex adapter: you may read your own bytes back first ("echo"); discard
        #   exactly len(packet) bytes if your adapter does that (find out on the hardware).
        # - Read the 6-byte minimum status, then the remainder from its LEN byte; decode_status.
        # - Retry once on timeout / bad checksum; raise on error bits (Status.errors).
        # - Broadcast and SYNC_WRITE never reply: return None without reading.
        # TODO: implement on the real adapter
        raise NotImplementedError("ServoBus.transact needs the real adapter")

    def ping(self, servo_id: int) -> bool:
        """True if the servo answers a PING."""
        # TODO: implement (wrapper over transact)
        raise NotImplementedError

    def read_u16(self, servo_id: int, address: int) -> int:
        """Read a 2-byte register (little-endian)."""
        # TODO: implement (READ of 2 bytes, then from_u16_le on the status params)
        raise NotImplementedError

    def write_u8(self, servo_id: int, address: int, value: int):
        """Write a 1-byte register. No reply expected for BROADCAST_ID."""
        # TODO: implement
        raise NotImplementedError

    def write_u16(self, servo_id: int, address: int, value: int):
        """Write a 2-byte register. No reply expected for BROADCAST_ID."""
        # TODO: implement
        raise NotImplementedError

    def torque(self, servo_id: int, enable: bool):
        """Enable / disable torque (Reg.TORQUE_ENABLE)."""
        # TODO: implement
        raise NotImplementedError

    def set_goal_ticks(self, goals: dict):
        """goals: {servo_id: ticks}. One SYNC_WRITE, no reply."""
        # Design: sync_write on Reg.GOAL_POSITION, 2 bytes per servo, expect_reply=False.
        # TODO: implement
        raise NotImplementedError

    def read_positions(self, ids=tuple(JOINT_IDS.values())) -> dict:
        """{servo_id: present position ticks}."""
        # Design: start with one READ per servo (simple, 6 round trips); later replace with a
        # single SYNC_READ round trip (TODO list) and measure the difference.
        # TODO: implement
        raise NotImplementedError

    def set_joint_targets(self, q_rad: dict):
        """q_rad: {joint name: rad}. Callers must pass targets through safety.validate_command first."""
        # Design: map names -> IDs (JOINT_IDS) and rad -> ticks (raw_from_joint_rad), then
        # set_goal_ticks. This method does not check limits: safety is the caller's job.
        # TODO: implement
        raise NotImplementedError

    def round_trip_time(self, servo_id: int = 1, n: int = 100) -> float:
        """Mean seconds per PING round trip (Python path number for the C++ comparison)."""
        # Design: time n pings with time.perf_counter and divide; report alongside the C++ number.
        # TODO: implement
        raise NotImplementedError

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - ping(1) == FF FF 01 02 01 FB   (checksum ~(1+2+1) = 0xFB)
# - read(3, PRESENT_POSITION, 2): 8 bytes, LEN = 4, checksum by hand
# - write(1, GOAL_POSITION, u16_le(2048)) == FF FF 01 05 03 2A 00 08 CHK (little-endian 2048 = 00 08)
# - sync_write of 2 servos x 2 bytes: ID = 0xFE, LEN = 2 + 6 + 2, total 14 bytes; wrong data length raises
# - decode_status with 2 garbage bytes before a valid reply -> correct Status and consumed == len(reply)
# - bad checksum and truncated packet -> (None, 0)
# - Status(1, 0b100100, b"").errors == ["overheat", "overload"]
# - ticks_to_rad(2048) == 0, ticks_to_rad(3072) == pi/2; rad_to_ticks(ticks_to_rad(t)) == t; rad_to_ticks(100) == 4095
# - raw_from_joint_rad(name, joint_rad_from_raw(name, 1500)) == 1500 (also with a non-default calibration)
# The same packet vectors must pass in gambit_cpp/tests/test_sts_protocol.cpp and on the STM32.
# Example shape of a test:
# assert ping(1) == bytes([0xFF, 0xFF, 0x01, 0x02, 0x01, 0xFB])
print("Self-check: not written yet (see plan above)")
print("servo_bus OK")